<div style="color: red; font-size: 40px; text-align: center;">TRANSFORMERS</div>

<div style="color: green; font-size: 30px;">1. Using Raw Audio</div>

**NOTE**
- This is just the skeleton of the model.
- I found that training a transformer demands too much compute that my lone CPU core struggles to deliver.
- It would take a minimum of 6 days to train over 10 epochs.
- Opted for google colab to delegate future training and most of the compute.
- Admittedly, i think i'll stick to pre-trained transformers to ease my work.
- Not to say that this has been a waste: I've learnt a lot of basics, just as intended!

In [ ]:
# Import relevant libraries
import os
import pandas as pd
import numpy as np

from datasets import Dataset
from transformers import Wav2Vec2Processor, Wav2Vec2ForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import classification_report, accuracy_score

import torch
import torchaudio

In [ ]:
# Process data. End up with a list of (path, label), convert to df
EMOTION_MAP = {
    "ANG": "angry", "NEU": "neutral", "FEA": "fearful",
    "SAD": "sad", "HAP": "happy", "DIS": "disgust"
}

data = []
audio_dir = "../data"
# Extract data
for filename in os.listdir(audio_dir):
    if filename.endswith('.wav'):
        emotion = filename.split("_")[2]
        emotion = EMOTION_MAP[emotion]
        path = os.path.join(audio_dir, filename)
        data.append((path, emotion))

df = pd.DataFrame(data, columns=["path", "label"])

In [ ]:
# Save df
df.to_pickle("df.pkl")

# Load processor
processor = Wav2Vec2Processor.from_pretrained("facebook/wav2vec2-base")

# Convert pandas to HF dataset
dataset = Dataset.from_pandas(df)

# Preprocessing function
def preprocess(example):
    waveform, sample_rate = torchaudio.load(example["path"])
    waveform = torchaudio.functional.resample(waveform, sample_rate, 16000)
    example["input_values"] = processor(waveform.squeeze().numpy(), sampling_rate=16000).input_values[0]
    example["label"] = label2id[example["label"]]
    return example

label2id = {label: i for i, label in enumerate(df["label"].unique())}
id2label = {i: label for label, i in label2id.items()}

dataset = dataset.map(preprocess, remove_columns=["path"])

In [ ]:
# Load model
model = Wav2Vec2ForSequenceClassification.from_pretrained("facebook/wav2vec2-base", num_labels=6, label2id=label2id, id2label=id2label)

# Split dataset
dataset = dataset.train_test_split(test_size=0.2)

# Training arguments
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    learning_rate=1e-4,
    per_device_eval_batch_size=8,
    per_device_train_batch_size=8,
    num_train_epochs=10,
    save_steps=500,
    logging_steps=100,
    load_best_model_at_end=True,
    metric_for_best_model='accuracy'
)

# Metrics
def compute_metrics(val_pred):
    logits, labels = val_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "Accuracy": accuracy_score(labels, preds)
    }

# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    tokenizer=processor.feature_extractor,
    compute_metrics=compute_metrics
)
trainer.train()

In [ ]:
# Evaluate model
preds = trainer.predict(dataset["test"])
print(classification_report(preds.label_ids, preds.predictions.argmax(-1), target_names=list(label2id.keys())))

In [ ]:
# Gradio inference app
import gradio as gr

def predict_emotion(audio):
    inputs = processor(audio['array'], sampling_rate=16000, return_tensors='pt')
    with torch.no_grad():
        logits = model(**inputs).logits
    prediction = torch.argmax(logits, dim=-1).items()
    return id3label[prediction]

interface = gr.Interface(fn=predict_emotion, inputs=gr.Audio(sources="microphone", type="numpy"), outputs="text")
interface.launch()